### Model 2: Neighbourhood-Based Features

This section constructs neighbourhood-based features using the **k-nearest neighbours (kNN)** algorithm based on latitude and longitude.

The kNN model is fitted using the **training coordinates only**, so that neighbourhood statistics are computed solely from training data and data leakage is avoided.

A helper function `neighbour_summary_feature()` is defined to compute summary statistics of neighbour prices.  
For each observation, the prices of its **K nearest neighbours** are used to calculate five statistics: **mean, median, minimum, maximum, and range**.

When generating features for the training set, the observation itself is excluded from its neighbour set.

In [ ]:
# Model 2: Add neighbourhood-based features

from sklearn.neighbors import NearestNeighbors

K = 7  # Number of nearest neighbours used to construct neighbourhood features


#   Fit kNN using training coordinates only, this ensures neighbourhood features are constructed 
#   solely from training data and avoids data leakage.
knn = NearestNeighbors(n_neighbors=K + 1)
knn.fit(X_train_raw[['Latitude', 'Longitude']])

# Align y_train with X_train_raw so neighbour indices correctly map to prices
y_train_aligned = y_train.loc[X_train_raw.index].to_numpy()



# Define a function to compute neighbourhood price statistics
# 这部分是用来计算第一个图所需要的数据
def neighbour_summary_feature(query_df, summary="mean", exclude_self=False):
    # Find nearest neighbours based on geographic coordinates
    _, inds = knn.kneighbors(query_df[['Latitude', 'Longitude']])

    values = []

    # Loop through each sample and compute the summary statistic
    for row_i, idxs in enumerate(inds):

        if exclude_self:
            # Remove the sample itself if it appears in neighbours
            idxs = idxs[idxs != row_i]

            # Keep exactly K neighbours
            idxs = idxs[:K]

        else:
            # Validation/test samples are not part of training set
            # so we directly take the first K neighbours
            idxs = idxs[:K]

        # Retrieve neighbour prices
        neigh_prices = y_train_aligned[idxs]

        if summary == "mean":
            values.append(np.mean(neigh_prices))

        elif summary == "median":
            values.append(np.median(neigh_prices))

        elif summary == "min":
            values.append(np.min(neigh_prices))

        elif summary == "max":
            values.append(np.max(neigh_prices))

        elif summary == "range":
            values.append(np.max(neigh_prices) - np.min(neigh_prices))

        else:
            raise ValueError("summary must be one of: mean, median, min, max, range")

    return np.array(values)



# Neighbourhood mean price 
neigh_price_mean_train = neighbour_summary_feature(X_train_raw, summary="mean", exclude_self=True)
neigh_price_mean_val   = neighbour_summary_feature(X_val_raw, summary="mean", exclude_self=False)
neigh_price_mean_test  = neighbour_summary_feature(X_test_raw, summary="mean", exclude_self=False)

# Neighbourhood median price 
neigh_price_median_train = neighbour_summary_feature(X_train_raw, summary="median", exclude_self=True)
neigh_price_median_val   = neighbour_summary_feature(X_val_raw, summary="median", exclude_self=False)
neigh_price_median_test  = neighbour_summary_feature(X_test_raw, summary="median", exclude_self=False)

# Neighbourhood minimum price 
neigh_price_min_train = neighbour_summary_feature(X_train_raw, summary="min", exclude_self=True)
neigh_price_min_val   = neighbour_summary_feature(X_val_raw, summary="min", exclude_self=False)
neigh_price_min_test  = neighbour_summary_feature(X_test_raw, summary="min", exclude_self=False)

# Neighbourhood maximum price 
neigh_price_max_train = neighbour_summary_feature(X_train_raw, summary="max", exclude_self=True)
neigh_price_max_val   = neighbour_summary_feature(X_val_raw, summary="max", exclude_self=False)
neigh_price_max_test  = neighbour_summary_feature(X_test_raw, summary="max", exclude_self=False)

# Neighbourhood price range 
neigh_price_range_train = neighbour_summary_feature(X_train_raw, summary="range", exclude_self=True)
neigh_price_range_val   = neighbour_summary_feature(X_val_raw, summary="range", exclude_self=False)
neigh_price_range_test  = neighbour_summary_feature(X_test_raw, summary="range", exclude_self=False)

### Baseline Validation Performance

A baseline neural network model is first trained using only the original features.  
Inputs are standardised using `StandardScaler`, and the **validation MSE** is computed as a reference for later comparisons.


In [ ]:
### 用于计算合并之前的baseline validation MSE的函数 
# 后面用到的baseline model都是这个，就不用繁琐的每次都写了
scaler_base = StandardScaler()

X_train_base = scaler_base.fit_transform(X_train_raw)
X_val_base = scaler_base.transform(X_val_raw)

baseline_model = MLPRegressor(
    hidden_layer_sizes=(128, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

#计算后续需要的baseline validation MSE
baseline_model.fit(X_train_base, y_train)
baseline_val_pred = baseline_model.predict(X_val_base)
baseline_val_mse = mean_squared_error(y_val, baseline_val_pred)

### Single Neighbourhood Feature Comparison

This section evaluates the five neighbourhood-based features separately.  
For each experiment, a single feature is added to the original dataset and the neural network model is retrained.

The function `run_extra_feature()` performs the following steps:
- add the new feature to the dataset
- standardise the inputs
- train the neural network
- compute the validation MSE

The results are collected in a table to compare the performance of different neighbourhood statistics.

In [ ]:
# Compare five single neighbourhood-based features
# Each experiment adds only one additional feature to the baseline model for comparison


def run_extra_feature(feature_name, f_train, f_val):
    # 1) Copy the original raw feature sets. This prevents modification of the original data
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    # 2) Add the new neighbourhood-based feature
    Xtr[feature_name] = f_train
    Xva[feature_name] = f_val

    # 3) Re-standardise the dataset after adding the new feature
    #    The scaler is fitted only on the training set to avoid data leakage
    sc = StandardScaler()

    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)

    # 4) Train a neural network model using the same architecture as the baseline model to ensure a fair comparison
    baseline_model.fit(Xtr_s, y_train)

    # 5) Compute validation MSE for the model with the additional feature

    baseline_val_pred = baseline_model.predict(Xva_s)

    val_mse = mean_squared_error(y_val, baseline_val_pred)
    return val_mse


# Run experiments for each neighbourhood feature separately

single_feature_results = []


# 1) Mean price of K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceMean",
    neigh_price_mean_train, neigh_price_mean_val
)
single_feature_results.append(("NeighbourPriceMean", val_mse))


# 2) Median price of K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceMedian",
    neigh_price_median_train, neigh_price_median_val
)
single_feature_results.append(("NeighbourPriceMedian", val_mse))


# 3) Minimum price among K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceMin",
    neigh_price_min_train, neigh_price_min_val
)
single_feature_results.append(("NeighbourPriceMin", val_mse))


# 4) Maximum price among K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceMax",
    neigh_price_max_train, neigh_price_max_val
)
single_feature_results.append(("NeighbourPriceMax", val_mse))


# 5) Price range among K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceRange",
    neigh_price_range_train, neigh_price_range_val
)
single_feature_results.append(("NeighbourPriceRange", val_mse))


# Convert results into a DataFrame for easy comparison and sort by validation MSE
single_feature_results_df = pd.DataFrame(single_feature_results, columns=["Feature", "Val MSE"])
single_feature_results_df = single_feature_results_df.sort_values("Val MSE").reset_index(drop=True)

print(single_feature_results_df)

### Neighbourhood Feature Combination Experiments

In addition to evaluating individual neighbourhood features, this section tests **combinations of multiple neighbourhood statistics**.

A helper function is used to add selected neighbourhood features to the original dataset, re-standardise the inputs, train the neural network, and compute the **validation MSE**.

All possible **1-feature, 2-feature, and 3-feature combinations** are evaluated.  
The results are collected in a table to analyse how the number and type of neighbourhood features affect model performance.

In [ ]:
# Evaluate combinations of neighbourhood-based features
# This section tests multiple feature combinations

from itertools import combinations
import pandas as pd


# Store neighbourhood features for training and validation sets
feature_pool_train = {
    "Mean": neigh_price_mean_train,
    "Median": neigh_price_median_train,
    "Min": neigh_price_min_train,
    "Max": neigh_price_max_train,
    "Range": neigh_price_range_train
}

feature_pool_val = {
    "Mean": neigh_price_mean_val,
    "Median": neigh_price_median_val,
    "Min": neigh_price_min_val,
    "Max": neigh_price_max_val,
    "Range": neigh_price_range_val
}



# Add multiple neighbourhood features and compute validation MSE
def run_multi_features(feature_dict_train, feature_dict_val):

    # Copy original datasets to avoid modifying them
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    # Add selected neighbourhood features
    for col, arr in feature_dict_train.items():
        Xtr[col] = arr

    for col, arr in feature_dict_val.items():
        Xva[col] = arr

    # Re-standardise inputs
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)

    # Train model and compute validation MSE
    baseline_model.fit(Xtr_s, y_train)
    baseline_val_pred  = baseline_model.predict(Xva_s)

    return mean_squared_error(y_val, baseline_val_pred)


# Evaluate different feature combinations
def evaluate_feature_combinations(
    feature_pool_train,
    feature_pool_val,
    combo_sizes=None,
    selected_combos=None
):

    feature_names = list(feature_pool_train.keys())
    combo_neighbour_results = []

    # Generate combinations automatically if not specified
    if selected_combos is not None:
        combos_to_run = selected_combos

    else:
        # If no specific combinations are given,
        # generate combinations automatically
        if combo_sizes is None:
            combo_sizes = list(range(1, len(feature_names) + 1))

        combos_to_run = []

        # Generate combinations of size k
        for k in combo_sizes:
            combos_to_run.extend(combinations(feature_names, k))


    # Run experiment for each combination
    for combo in combos_to_run:

        combo = tuple(combo)

        # Extract the corresponding feature arrays
        feature_dict_train = {name: feature_pool_train[name] for name in combo}
        feature_dict_val   = {name: feature_pool_val[name] for name in combo}

        val_mse = run_multi_features(
            feature_dict_train,
            feature_dict_val,
        )

        # Store results
        combo_neighbour_results.append({
            "combo_name": " + ".join(combo),
            "num_features": len(combo),
            "val_mse": val_mse
        })



    # Sort results by feature count and validation MSE
    return pd.DataFrame(combo_neighbour_results).sort_values(
        by=["num_features", "val_mse"],
        ascending=[True, True]
    ).reset_index(drop=True)


# Run experiments for 1-feature, 2-feature, and 3-feature combinations
combo_results_df = evaluate_feature_combinations(
    feature_pool_train,
    feature_pool_val,
    combo_sizes=[1, 2, 3]
)

# Display the results
display(combo_results_df)

### Plot Validation MSE vs Number of Neighbourhood Features

This plot shows how validation MSE changes as the number of neighbourhood features increases.

Each scatter point represents the validation MSE of a specific feature combination.  
The lines show the **minimum, average, and maximum validation MSE** for each number of neighbourhood features.

The dashed horizontal line indicates the **baseline model performance**, allowing us to compare whether adding neighbourhood features improves prediction accuracy.

In [ ]:
# Plot: Validation MSE vs Number of Neighbourhood Features

import matplotlib.pyplot as plt
import numpy as np


# Compute summary statistics (min, mean, max) for each feature count
summary_df = (
    combo_results_df
    .groupby("num_features")["val_mse"]    
    .agg(["min", "mean", "max"])          
    .reset_index()                      
)


# Scatter plot for all feature combinations
plt.figure(figsize=(10,6))
np.random.seed(26)

plt.scatter(
    combo_results_df["num_features"],   
    combo_results_df["val_mse"],        
    alpha=0.7,
    label="All combinations"
)


# Plot min / average / max validation MSE
plt.plot(
    summary_df["num_features"],
    summary_df["min"],
    marker="o",
    label="Minimum"
)

plt.plot(
    summary_df["num_features"],
    summary_df["mean"],
    marker="o",
    label="Average"
)

plt.plot(
    summary_df["num_features"],
    summary_df["max"],
    marker="o",
    label="Maximum"
)


# Baseline model reference line
plt.axhline(
    y=baseline_val_mse,
    linestyle="--",
    label="Baseline model"
)


# Figure formatting
plt.xlabel("Number of Neighbourhood Features")   
plt.ylabel("Validation MSE")                     
plt.title("Validation MSE for Feature Combinations")  

plt.xticks(sorted(combo_results_df["num_features"].unique()))
plt.legend()                 
plt.grid(True, alpha=0.3)    

plt.show()

From the graph, the lowest validation MSE occurs when **only one neighbourhood feature** is added. Therefore, in the following experiments we only consider adding **one additional neighbourhood feature**.

### Validation MSE for Different Values of \(k\)

This section evaluates how the choice of **\(k\)** affects the usefulness of neighbourhood-based features.

For each value of \(k\), neighbourhood features are recomputed using the corresponding \(k\)-nearest neighbours.  
Each feature is then added to the original dataset individually, and the model is retrained to obtain the **validation MSE**.

The results are stored in a table for comparing how different neighbourhood statistics perform across different values of \(k\).

In [ ]:
# Generate results_k_feature_df for Figure 2 (Validation MSE)

from sklearn.neighbors import NearestNeighbors

k_values = [3, 5, 7, 10, 15]

def compute_features_for_k(K):
    knn = NearestNeighbors(n_neighbors=K + 1)
    knn.fit(X_train_raw[['Latitude', 'Longitude']])

    y_train_aligned = y_train.loc[X_train_raw.index].to_numpy()

    # Compute neighbour indices for training and validation sets
    _, idx_train = knn.kneighbors(X_train_raw[['Latitude', 'Longitude']])
    _, idx_val = knn.kneighbors(X_val_raw[['Latitude', 'Longitude']])

    # Remove self from training neighbours
    idx_train = idx_train[:, 1:K+1]
    idx_val = idx_val[:, :K]

    # Convert neighbour indices to price matrices
    train_prices = y_train_aligned[idx_train]
    val_prices = y_train_aligned[idx_val]

    # Compute neighbourhood summary statistics for training and validation sets
    neigh_price_train_k = train_prices.mean(axis=1)
    neigh_price_val_k   = val_prices.mean(axis=1)

    neigh_price_median_train_k = np.median(train_prices, axis=1)
    neigh_price_median_val_k   = np.median(val_prices, axis=1)

    neigh_price_min_train_k = train_prices.min(axis=1)
    neigh_price_min_val_k   = val_prices.min(axis=1)

    neigh_price_max_train_k = train_prices.max(axis=1)
    neigh_price_max_val_k   = val_prices.max(axis=1)

    neigh_price_range_train_k = neigh_price_max_train_k - neigh_price_min_train_k
    neigh_price_range_val_k   = neigh_price_max_val_k - neigh_price_min_val_k

    return {
        "Mean":   (neigh_price_train_k, neigh_price_val_k),
        "Median": (neigh_price_median_train_k, neigh_price_median_val_k),
        "Min":    (neigh_price_min_train_k, neigh_price_min_val_k),
        "Max":    (neigh_price_max_train_k, neigh_price_max_val_k),
        "Range":  (neigh_price_range_train_k, neigh_price_range_val_k),
    }


def run_one_feature(feature_name, f_train, f_val):
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    # Add one neighbourhood feature
    Xtr[feature_name] = f_train
    Xva[feature_name] = f_val

    # Re-standardise after adding the new feature
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)

    # Train and evaluate
    baseline_model.fit(Xtr_s, y_train)
    val_pred = baseline_model.predict(Xva_s)

    return mean_squared_error(y_val, val_pred)

k_feature_results = []

for K in k_values:
    feature_data = compute_features_for_k(K)

    for feature_name, (f_train, f_val) in feature_data.items():
        val_mse = run_one_feature(feature_name, f_train, f_val)

        k_feature_results.append({
            "k": K,
            "feature": feature_name,
            "val_mse": val_mse
        })

k_feature_results_df = pd.DataFrame(k_feature_results)

print(k_feature_results_df)

### Plot Validation MSE vs Number of Neighbours (k)

This figure shows how the validation MSE changes with different values of **\(k\)** for each single neighbourhood feature.

Each line represents one neighbourhood statistic (mean, median, min, max, range).  
The dashed line indicates the **baseline model performance**, which serves as a reference to evaluate whether neighbourhood features improve the model.

This plot helps identify the most suitable value of \(k\) for constructing neighbourhood-based features.

In [ ]:
# Figure 2: Validation MSE vs Number of Neighbours (k)
# Single neighbourhood features

plt.figure(figsize=(10, 6))

# Plot validation MSE curves for each single feature
for feature_name in k_feature_results_df["feature"].unique():

    subset = k_feature_results_df[
        k_feature_results_df["feature"] == feature_name
    ].sort_values("k")

    plt.plot(
        subset["k"],
        subset["val_mse"],
        marker="o",
        label=feature_name
    )

# Plot baseline model performance as a reference line
plt.axhline(
    y=baseline_val_mse,
    linestyle="--",
    label="Baseline model"
)

# Figure formatting
plt.xlabel("Number of Neighbours (k)")
plt.ylabel("Validation MSE")
plt.title("Validation MSE vs Number of Neighbours (Single Neighbourhood Features)")

plt.xticks(sorted(k_feature_results_df["k"].unique()))
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True, alpha=0.3)

plt.show()

From this plot, the lowest validation MSE is achieved when **k = 10** and the additional neighbourhood feature is the **mean price of neighbouring houses**.

### Final Enhanced Model and Test Set Comparison

In this section, the original training and validation sets are combined to form a final training dataset.

Using the selected best neighbourhood feature(s) and the optimal value of **\(k\)**, the neighbourhood statistics are recomputed on the final training set and added to the input features.  
The enhanced neural network model is then retrained on the combined data and evaluated on the test set.

Finally, the **test MSE** of the enhanced model is compared with that of the baseline model to assess whether neighbourhood-based features improve the final prediction performance.

In [ ]:
# from sklearn.neighbors import NearestNeighbors

# The above import is already present at the top of the notebook,
# but just for convienience for skipping the plotting cells and 
# calculate the final test MSE after selecting the best feature combination


# 1) Combine original train and validation sets
X_train_final_raw = pd.concat([X_train_raw, X_val_raw], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

# Align indices
X_train_final_raw = X_train_final_raw.sort_index()
y_train_final = y_train_final.loc[X_train_final_raw.index]

# 2) Fit kNN on the new final training set
best_K = 10
knn_final = NearestNeighbors(n_neighbors=best_K + 1)
knn_final.fit(X_train_final_raw[['Latitude', 'Longitude']])

y_train_final_aligned = y_train_final.to_numpy()

# 3) Define function to compute neighbourhood summary features
def neighbour_summary_feature_final(query_df, summary="mean", exclude_self=False):
    _, inds = knn_final.kneighbors(query_df[['Latitude', 'Longitude']])
    values = []

    for row_i, idxs in enumerate(inds):
        if exclude_self:
            idxs = idxs[idxs != row_i]
            idxs = idxs[:best_K]
        else:
            idxs = idxs[:best_K]

        neigh_prices = y_train_final_aligned[idxs]

        if summary == "mean":
            values.append(np.mean(neigh_prices))
        elif summary == "median":
            values.append(np.median(neigh_prices))
        elif summary == "min":
            values.append(np.min(neigh_prices))
        elif summary == "max":
            values.append(np.max(neigh_prices))
        elif summary == "range":
            values.append(np.max(neigh_prices) - np.min(neigh_prices))
        else:
            raise ValueError("summary must be one of: mean, median, min, max, range")

    return np.array(values)

# 4) Compute only the selected best features
feature_map = {
    "Mean": "mean",
    "Median": "median",
    "Min": "min",
    "Max": "max",
    "Range": "range"
}

best_features = ["Mean"]   # 输入最终选出来的组合

train_final_feature_values = {}
test_feature_values = {}

for feat in best_features:
    summary_name = feature_map[feat]
    train_final_feature_values[feat] = neighbour_summary_feature_final(
        X_train_final_raw, summary=summary_name, exclude_self=True
    )
    test_feature_values[feat] = neighbour_summary_feature_final(
        X_test_raw, summary=summary_name, exclude_self=False
    )


# 5) Add selected feature(s) to the final dataset
X_train_enh_final_raw = X_train_final_raw.copy()
X_test_enh_final_raw = X_test_raw.copy()

for feat in best_features:
    col_name = f"NeighbourPrice{feat}"
    X_train_enh_final_raw[col_name] = train_final_feature_values[feat]
    X_test_enh_final_raw[col_name] = test_feature_values[feat]


# 6) Train the final enhanced model
scaler_enh_final = StandardScaler()

X_train_enh_final = pd.DataFrame(
    scaler_enh_final.fit_transform(X_train_enh_final_raw),
    columns=X_train_enh_final_raw.columns,
    index=X_train_enh_final_raw.index
)

X_test_enh_final = pd.DataFrame(
    scaler_enh_final.transform(X_test_enh_final_raw),
    columns=X_test_enh_final_raw.columns,
    index=X_test_enh_final_raw.index
)

model_enh_final = MLPRegressor(
    hidden_layer_sizes=(128, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

model_enh_final.fit(X_train_enh_final, y_train_final)
test_pred_enh_final = model_enh_final.predict(X_test_enh_final)
enhanced_test_mse_final = mean_squared_error(y_test, test_pred_enh_final)

# 7) Final comparison
print("Final Baseline Test MSE:", baseline_test_mse_final)
print("Final Enhanced Test MSE:", enhanced_test_mse_final)
print("Delta Test MSE (Enhanced - Baseline):", enhanced_test_mse_final - baseline_test_mse_final)